# 커스텀 예산 미들웨어

모델 호출 수뿐 아니라 예상비용 상한을 호출 직전에 어떻게 검사할까요?

## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/05-커스텀-예산-미들웨어.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.

## 패키지 버전 확인

In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비

In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## Before: 비용 검사 없는 두 호출

In [3]:
# 이 셀은 Before: 비용 검사 없는 두 호출 작업을 수행하고 확인할 값을 출력합니다.
plain=ScriptedModel(replies=[AIMessage("첫 응답"),AIMessage("둘째 응답")])
plain.invoke("첫 호출"); plain.invoke("둘째 호출")
print("calls:",plain.index)
assert plain.index==2

calls: 2


## After: 예산 미들웨어 작성

In [4]:
# 이 셀은 After: 예산 미들웨어 작성 작업을 수행하고 확인할 값을 출력합니다.
class BudgetExceeded(RuntimeError): pass
class BudgetMiddleware(AgentMiddleware):
    def __init__(self, max_cost_usd: float, estimated_cost_per_call: float) -> None:
        self.max_cost_usd=max_cost_usd; self.estimated=estimated_cost_per_call; self.reserved=0.0
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        next_cost=self.reserved+self.estimated
        if next_cost > self.max_cost_usd: raise BudgetExceeded("이 실행의 예상비용 상한을 초과합니다.")
        self.reserved=next_cost
        return handler(request)

## 첫 호출 허용과 두 번째 차단

In [5]:
# 이 셀은 첫 호출 허용과 두 번째 차단 작업을 수행하고 확인할 값을 출력합니다.
budget=BudgetMiddleware(max_cost_usd=0.006,estimated_cost_per_call=0.004)
guarded=ScriptedModel(replies=[AIMessage("첫 응답"),AIMessage("둘째 응답")])
agent=create_agent(guarded,tools=[],middleware=[budget])
first=agent.invoke({"messages":[{"role":"user","content":"첫 호출"}]})
try:
    agent.invoke({"messages":[{"role":"user","content":"둘째 호출"}]})
except BudgetExceeded as error:
    print(type(error).__name__,str(error))
print("model_calls:",guarded.index,"reserved:",budget.reserved)
assert guarded.index==1 and budget.reserved==0.004

BudgetExceeded 이 실행의 예상비용 상한을 초과합니다.
model_calls: 1 reserved: 0.004


## 비용 기록

In [6]:
# 이 셀은 비용 기록 작업을 수행하고 확인할 값을 출력합니다.
print({"max_cost_usd":budget.max_cost_usd,"reserved_cost_usd":budget.reserved,"actual_api_cost_usd":0.0})
assert budget.reserved < 1.0

{'max_cost_usd': 0.006, 'reserved_cost_usd': 0.004, 'actual_api_cost_usd': 0.0}
